# Variational Algorithms: Hamiltonians & Observables Lab

*Module 5 assignment.* Work through the sections below. Add your written answers in the markdown
cells and your code in the cells provided (you can insert more cells as needed).

**Everything required in this lab runs on a local simulator.** No IBM Quantum account, no queue,
and no QPU time is needed. Only the optional bonus at the end touches real hardware.


# Objective

In Module 7 you will run a complete algorithm that somebody else wrote. This lab is about the
machinery *underneath* those algorithms.

Almost every near-term quantum algorithm that people actually propose to run -- VQE for chemistry,
QAOA for optimization, quantum machine learning -- is a **variational algorithm**. They all share
one shape:

1. Prepare a state with a circuit that has tunable knobs (**parameters**).
2. Measure the **expectation value** of an observable, giving one number: the *energy*.
3. Hand that number to a classical optimizer, which turns the knobs and asks again.

Step 2 is the interface between the quantum computer and everything else. A quantum computer does
not hand you a wavefunction; it hands you *estimates of expectation values*. If you understand
observables, Hamiltonians, and expectation values, you understand what these algorithms are
actually doing, and you can read a VQE or QAOA paper without the notation stopping you.

By the end of this lab you will have built a working variational energy minimization **by hand**,
with no optimizer library, and checked it against the exact answer from linear algebra.

### Reference material

- [Variational Algorithms](https://quantum.cloud.ibm.com/learning/en/courses/variational-algorithm-design/variational-algorithms) (IBM Quantum Learning)
- [Variational Quantum Eigensolver (VQE)](https://quantum.cloud.ibm.com/learning/en/courses/quantum-diagonalization-algorithms/vqe) (IBM Quantum Learning)
- [Introduction to primitives](https://quantum.cloud.ibm.com/docs/en/guides/primitives) -- the `Estimator` / `Sampler` split (IBM docs)

### Setup

Run the cell below first. It only needs `qiskit`, `numpy`, and `matplotlib`, all of which you
already have from earlier modules.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator

# The Estimator is the primitive that returns EXPECTATION VALUES.
# (The Sampler, which you have used before, returns measurement counts instead.)
# Deliberately NOT seeded: Part 2 needs genuinely fresh randomness each call to
# show shot noise. Parts 1, 3 and 4 ask for exact values, so they are unaffected.
estimator = StatevectorEstimator()


def expectation(circuit, observable, precision=0.0):
    """Return <observable> for the state that `circuit` prepares.

    precision=0.0 means the exact value (infinite shots). A positive precision
    simulates the statistical error you would get from a finite number of shots.
    """
    if isinstance(observable, str):
        observable = SparsePauliOp(observable)
    result = estimator.run([(circuit, observable)], precision=precision).result()
    return float(np.asarray(result[0].data.evs).ravel()[0])


print("Setup complete. Qiskit is ready.")


---
# Part 1: Observables and expectation values

An **observable** is a physical quantity you can measure. In quantum computing we write observables
as Pauli operators. The two you will use constantly:

- $Z$ asks *"is this qubit 0 or 1?"* It has eigenvalue $+1$ for $|0\rangle$ and $-1$ for $|1\rangle$.
- $X$ asks the same question **in the Hadamard basis**: $+1$ for $|+\rangle$, $-1$ for $|-\rangle$.

The **expectation value** $\langle \psi | Z | \psi \rangle$, written $\langle Z \rangle$, is the
*average* value you would get if you measured $Z$ many times on freshly prepared copies of
$|\psi\rangle$. It is a single real number between $-1$ and $+1$.

This is worth pausing on: **an expectation value is not a measurement outcome.** A single
measurement of $Z$ always gives you exactly $+1$ or $-1$. The expectation value is what those
outcomes average to, and it can be any number in between, including $0$ -- a value no single
measurement can ever return.


## Part 1a: Predict first

Before running any code, work these out by hand.

***Fill in the table below with your predicted values. For each of the three states, what is***
$\langle Z \rangle$ ***and*** $\langle X \rangle$***? Show your reasoning -- one line per entry
is enough.***

| State | $\langle Z \rangle$ (predicted) | $\langle X \rangle$ (predicted) |
|---|---|---|
| $\vert 0 \rangle$ | | |
| $\vert 1 \rangle$ | | |
| $\vert + \rangle = H\vert 0 \rangle$ | | |

*Hint: for* $\langle Z \rangle$ *on* $|+\rangle$*, remember that measuring* $|+\rangle$ *in the
computational basis gives 0 half the time and 1 half the time. What do* $+1$ *and* $-1$ *average to?*

**Your reasoning:**

*(write here)*


## Part 1b: Check with the Estimator

***Run the cell below and compare against your predictions. Did anything surprise you?***


In [ ]:
# Three single-qubit states
s0 = QuantumCircuit(1)                    # |0>
s1 = QuantumCircuit(1); s1.x(0)           # |1>
splus = QuantumCircuit(1); splus.h(0)     # |+>

print(f"{'state':>8}   {'<Z>':>7}   {'<X>':>7}")
for name, circ in (("|0>", s0), ("|1>", s1), ("|+>", splus)):
    print(f"{name:>8}   {expectation(circ, 'Z'):+7.3f}   {expectation(circ, 'X'):+7.3f}")


***Question.*** *For the state* $|+\rangle$ *you should find* $\langle Z \rangle = 0$ *but*
$\langle X \rangle = +1$. **Explain in your own words what that pair of numbers tells you about
the state.** *Why is it wrong to read* $\langle Z \rangle = 0$ *as "the qubit has no definite
value"? Be specific about what the choice of observable is really choosing.*

**Your answer:**

*(write here)*


---
# Part 2: Where the cost comes from -- shot noise

The exact expectation values above came from a simulator that has access to the full statevector.
**A real quantum computer cannot do that.** It prepares the state, measures, and gets one outcome.
To estimate $\langle X \rangle$ it must repeat the whole thing thousands of times and average.
Each repetition is a **shot**.

That averaging has a statistical error, and the error shrinks only as
$1/\sqrt{\text{shots}}$. This is the single most important practical fact about running
variational algorithms, so it is worth measuring yourself.

***Run the cell below. It estimates*** $\langle X \rangle$ ***for*** $|+\rangle$ ***many times at
each shot count, and reports how much the answers scatter.***

> Your exact numbers will differ slightly every time you run this cell, and that is the whole
> point -- the randomness is real. The *trend* is what matters, not the individual values.


In [ ]:
shot_counts = [100, 400, 1600, 6400, 25600]
repeats = 40
exact_value = expectation(splus, "X")     # = +1.000

spreads = []
for shots in shot_counts:
    # A finite number of shots gives a statistical error of roughly 1/sqrt(shots).
    trials = [expectation(splus, "X", precision=1/np.sqrt(shots)) for _ in range(repeats)]
    spread = float(np.std(trials))
    spreads.append(spread)
    print(f"shots = {shots:6d}   spread of estimates = {spread:.4f}   1/sqrt(shots) = {1/np.sqrt(shots):.4f}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(shot_counts, spreads, "o-", label="measured spread")
ax.loglog(shot_counts, [1/np.sqrt(s) for s in shot_counts], "--", label=r"$1/\sqrt{shots}$")
ax.set_xlabel("shots"); ax.set_ylabel("spread of the estimate (std. dev.)")
ax.set_title(r"Estimating $\langle X \rangle$: error vs shots")
ax.legend(); ax.grid(True, which="both", alpha=0.3)
plt.show()


***Question.*** *Suppose one estimate of the energy at your current precision is good enough,
and now you want* **one extra decimal digit** *of precision, i.e. 10x smaller error.*

1. ***How many times more shots does that take?*** *Read it off the scaling, and say why.*
2. *A variational algorithm needs a fresh energy estimate at* **every step** *of the optimizer, and
   a real Hamiltonian has many terms, each needing its own measurement.* ***Given that, explain in
   your own words why "just run more shots" is not a free fix on real hardware.***

**Your answer:**

*(write here)*


---
# Part 3: Hamiltonians as sums of Paulis

A **Hamiltonian** $H$ is the observable whose expectation value is the *energy* of a state. It is
the thing a variational algorithm tries to minimize.

Any Hamiltonian you can run on a quantum computer gets written as a weighted sum of Pauli strings:

$$ H = \sum_k c_k P_k $$

where each $P_k$ is something like $ZZ$, $XI$, or $IZ$, and each $c_k$ is a real number. Qiskit
stores this with `SparsePauliOp`. Because expectation values are linear, the energy is just the
weighted sum of the individual Pauli expectation values:

$$ \langle H \rangle = \sum_k c_k \langle P_k \rangle $$

That linearity is what makes this practical: the machine measures each Pauli term separately, and
you add up the results.

Here is a small two-qubit Ising Hamiltonian:

$$ H_{\text{Ising}} = 1.0\, ZZ + 0.5\, ZI + 0.2\, IZ $$

***Before running the next cell:*** *this Hamiltonian is built only from* $Z$ *operators, so every
computational basis state* $|00\rangle, |01\rangle, |10\rangle, |11\rangle$ *has a definite
energy.* ***Which basis state do you predict has the lowest energy? Work it out by hand*** *(recall
that* $Z|0\rangle = +|0\rangle$ *and* $Z|1\rangle = -|1\rangle$*).*

**Your prediction and reasoning:**

*(write here)*


In [ ]:
H_ising = SparsePauliOp.from_list([
    ("ZZ", 1.0),
    ("ZI", 0.5),
    ("IZ", 0.2),
])
print("H_ising =")
print(H_ising)

# Because H is diagonal here, each basis state's energy sits on the diagonal of the matrix.
diagonal = np.real(np.diag(H_ising.to_matrix()))
print("\nEnergy of each computational basis state:")
for i, energy in enumerate(diagonal):
    print(f"  |{i:02b}>   {energy:+.2f}")

# The exact answer from linear algebra, for comparison.
eigenvalues = np.linalg.eigvalsh(H_ising.to_matrix())
print(f"\nEigenvalues: {np.round(eigenvalues, 4)}")
print(f"Ground-state energy: {eigenvalues.min():+.2f}  (state |{int(np.argmin(diagonal)):02b}>)")


***Question.*** *Was your prediction right?* ***Now build your own Hamiltonian:*** *change the
three coefficients so that a* **different** *basis state becomes the ground state. Write it in the
cell below, confirm with the same diagonal check, and* ***explain which coefficient you changed and
why it moved the minimum.***


In [ ]:
# Your Hamiltonian here.
my_H = SparsePauliOp.from_list([
    ("ZZ", 1.0),
    ("ZI", 0.5),
    ("IZ", 0.2),
])

diag = np.real(np.diag(my_H.to_matrix()))
for i, energy in enumerate(diag):
    print(f"  |{i:02b}>   {energy:+.2f}")
print(f"ground state: |{int(np.argmin(diag)):02b}>")


**Your explanation:**

*(write here)*


---
# Part 4: The variational loop, by hand

Now the real thing.

Below is the Hamiltonian for a **hydrogen molecule (H$_2$)** at its equilibrium bond length,
reduced to two qubits. This is a genuine quantum chemistry problem, and it is the standard first
benchmark for VQE:

$$ H = -1.0524\,II + 0.3979\,IZ - 0.3979\,ZI - 0.0113\,ZZ + 0.1809\,XX $$

Note the $XX$ term. The Hamiltonian is no longer diagonal, so the ground state is **not** a
computational basis state -- it is a superposition. That is exactly why this problem is interesting
and why you need a parameterized circuit to reach the answer.

Our **ansatz** (the trial circuit with a tunable knob) has a single parameter $\theta$:

- `X` on qubit 0 -- start from $|01\rangle$, the right particle number for this molecule
- `RY(θ)` on qubit 1 -- the knob
- `CNOT` from qubit 1 to qubit 0 -- creates the superposition the $XX$ term needs

***Run the two cells below.*** *The first defines everything; the second sweeps* $\theta$ *across a
grid and plots the energy landscape.*


In [ ]:
# The H2 Hamiltonian (2-qubit reduction, equilibrium bond length ~0.735 angstrom).
H2 = SparsePauliOp.from_list([
    ("II", -1.052373245772859),
    ("IZ",  0.39793742484318045),
    ("ZI", -0.39793742484318045),
    ("ZZ", -0.01128010425623538),
    ("XX",  0.18093119978423156),
])

# The exact answer, from diagonalizing the matrix classically.
# We can only do this because the problem is tiny -- that is the whole point of VQE.
exact_energy = float(np.linalg.eigvalsh(H2.to_matrix()).min())
print(f"Exact ground-state energy: {exact_energy:.6f} Hartree")

# The ansatz: one parameter.
theta = Parameter("theta")
ansatz = QuantumCircuit(2)
ansatz.x(0)
ansatz.ry(theta, 1)
ansatz.cx(1, 0)
print()
print(ansatz.draw())


In [ ]:
# Sweep theta and record the energy at each point.
# This IS the variational algorithm -- we are just doing the optimizer's job by brute force
# so you can see the whole landscape it would be searching.
grid = np.linspace(-np.pi, np.pi, 61)
result = estimator.run([(ansatz, H2, [[t] for t in grid])], precision=0.0).result()
energies = np.asarray(result[0].data.evs)

best = int(np.argmin(energies))
theta_star, e_star = grid[best], energies[best]

print(f"lowest energy found : {e_star:.6f} Ha   at theta = {theta_star:+.4f}")
print(f"exact               : {exact_energy:.6f} Ha")
print(f"difference          : {abs(e_star - exact_energy):.2e} Ha")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(grid, energies, lw=2, label=r"$E(\theta) = \langle \psi(\theta)|H|\psi(\theta)\rangle$")
ax.axhline(exact_energy, ls="--", c="crimson", label=f"exact = {exact_energy:.4f} Ha")
ax.plot(theta_star, e_star, "o", ms=10, c="darkgreen", label=f"minimum at {theta_star:.3f}")
ax.set_xlabel(r"$\theta$"); ax.set_ylabel("energy (Hartree)")
ax.set_title(r"H$_2$ energy landscape")
ax.legend(); ax.grid(alpha=0.3)
plt.show()


***Deliverables for Part 4.***

1. ***Report your numbers:*** *the minimum energy you found, the* $\theta$ *where it occurred, and
   how far it is from the exact value.*
2. ***Look at the shape of the curve.*** *A classical optimizer (COBYLA, gradient descent, ...) does
   not get to see this plot -- it can only ask for the energy at one* $\theta$ *at a time and decide
   where to step next.* ***Describe how an optimizer starting at*** $\theta = 3.0$ ***would find the
   minimum, and roughly how many energy evaluations you would expect it to need.***
3. ***Our grid used 61 points to find the minimum of a 1-parameter problem.*** *A realistic chemistry
   ansatz has dozens or hundreds of parameters.* ***Explain why sweeping a grid stops being possible,
   and what that implies about why the classical optimizer is essential.***

**Your answers:**

*(write here)*


---
# Part 5: Why this works -- the variational principle

Everything above rests on one theorem. For **any** state $|\psi\rangle$ you can possibly prepare,

$$ \langle \psi | H | \psi \rangle \;\geq\; E_{\text{ground}} $$

The energy of a trial state can never dip below the true ground-state energy. This is the
**variational principle**, and it is what makes the whole approach trustworthy: every number your
quantum computer reports is an *upper bound* on the answer, so "lower is better" is always the
right instruction, and you can never be fooled into thinking you did better than you did.

***Answer the following.***

1. *Look back at your Part 4 plot.* ***Is the curve ever below the red exact line? Should it be?
   What would you conclude if you ran a VQE and got an energy clearly below the known exact
   value?***
2. *Our ansatz had exactly one parameter, so it can only reach a one-dimensional family of states.*
   ***What happens if the true ground state is not in the set of states your ansatz can reach?***
   *Would the algorithm fail loudly, or quietly return something wrong? How would you even notice?*
   *(This property is called the* **expressibility** *of the ansatz.)*
3. *We found the exact answer here by diagonalizing a* $4 \times 4$ *matrix.* ***For a molecule with*** $n$
   ***qubits, how does the size of that matrix grow?*** *Use that to explain, in one or two sentences,
   why anyone would bother with VQE instead of just diagonalizing the matrix.*

**Your answers:**

*(write here)*


---
# Bonus (optional)

Either one of these. Optional, but each is a genuinely useful thing to have done once.

## Bonus A: A two-parameter landscape

Give the ansatz a second knob and map the landscape in 2D. This is what the optimizer is actually
walking around in -- and with two parameters you can still *see* it, which you cannot do with fifty.

***Build a 2-parameter ansatz, evaluate the energy on a grid, and plot a contour map. Mark the
minimum. How close does it get to the exact energy, and how does the landscape compare to the
1-parameter curve from Part 4?***


In [ ]:
# Bonus A starter -- a 2-parameter ansatz.
a, b = Parameter("a"), Parameter("b")
ansatz2 = QuantumCircuit(2)
ansatz2.ry(a, 0)
ansatz2.ry(b, 1)
ansatz2.cx(0, 1)

g = np.linspace(-np.pi, np.pi, 25)
points = [[x, y] for x in g for y in g]
res2 = estimator.run([(ansatz2, H2, points)], precision=0.0).result()
landscape = np.asarray(res2[0].data.evs).reshape(len(g), len(g))

print(f"best energy on this grid: {landscape.min():.6f} Ha   (exact {exact_energy:.6f} Ha)")

fig, ax = plt.subplots(figsize=(5.5, 4.5))
contour = ax.contourf(g, g, landscape.T, levels=20)
fig.colorbar(contour, label="energy (Ha)")
ax.set_xlabel("a"); ax.set_ylabel("b"); ax.set_title(r"H$_2$ energy landscape, 2 parameters")
plt.show()


**Your observations:**

*(write here)*

## Bonus B: Run it on real hardware

Take the best $\theta$ you found in Part 4 and evaluate the energy at that single point on a real
QPU, using `EstimatorV2` from `qiskit_ibm_runtime`. Compare it to the simulator value.

> **Warning: this consumes your QPU allotment.** Submit **one** circuit at **one** parameter value.
> Do not submit the whole 61-point sweep, and do not increase the qubit count. Your Module 7
> assignment also needs QPU time, so budget accordingly.

You will need to transpile the circuit for your chosen backend first (see the
[Module 3 transpilation material](https://quantum.cloud.ibm.com/docs/en/guides/transpile)).

***Report the hardware energy next to the simulator energy. How large is the gap, and which
direction does it go? Given the variational principle from Part 5, is a hardware result that comes
out* below *the exact energy evidence of a better answer, or evidence of something wrong?***

**Your results and answer:**

*(write here)*


---
## Your Work

Use the cells below for any additional code, notes, or plots.
